# DeepWeeds — Lab Day 2 trên Google Colab

**Runtime → Change runtime type → T4 GPU (hoặc GPU khác).** Chạy các ô từ trên xuống.

Notebook dùng bài làm trong `submissions/2A202602903_NguyenVanGiap/code`, tải dữ liệu chính thức, lưu kết quả vào Drive và tiếp tục từ epoch hoàn tất khi chạy lại. Cấu hình được chọn trên **validation**; test chỉ dùng ở chung kết. Không cần Python/uv/GPU trên máy cá nhân.

Upload kèm `colab_bundle.zip` đã chuẩn bị trong repo. Cách này dùng đúng mã vừa sửa kể cả khi chưa push GitHub. Bản chạy đầy đủ gồm 5 backbone, 3 trục công thức, 5 phương pháp suy luận bổ sung, chung kết và mốc với 3 seed. Có thể mất nhiều phiên GPU; checkpoint nằm trên Drive.


## 0. Nạp mã và thiết lập GPU

In [ ]:
import os, sys, json, shutil, subprocess, zipfile
from pathlib import Path
try:
    from google.colab import files, drive
except ImportError:
    raise RuntimeError("Notebook này dành cho Google Colab. Hãy mở tại https://colab.research.google.com/")

SOURCE = "upload"  # upload: colab_bundle.zip; github: mã đã push lên repo
REPO_DIR = Path("/content/deepweeds-lab")
if SOURCE == "upload":
    if not (REPO_DIR / "eval.py").exists():
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.endswith(".zip")]
        if len(candidates) != 1:
            raise ValueError("Chọn đúng một file colab_bundle.zip")
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(candidates[0]) as archive:
            for member in archive.infolist():
                if not (REPO_DIR / member.filename).resolve().is_relative_to(REPO_DIR.resolve()):
                    raise ValueError("Đường dẫn ZIP không hợp lệ")
            archive.extractall(REPO_DIR)
        del uploaded
elif SOURCE == "github":
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/Giappp/K4-DAY02-NguyenVanGiap-2A202602903.git", str(REPO_DIR)], check=True)
else:
    raise ValueError("SOURCE phải là upload hoặc github")

SUBMISSION_DIR = REPO_DIR / "submissions/2A202602903_NguyenVanGiap"
CODE_DIR = SUBMISSION_DIR / "code"
if not (CODE_DIR / "experiments.py").exists():
    raise RuntimeError("Repo/ZIP chưa chứa bản hoàn thiện. Upload colab_bundle.zip mới nhất.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(SUBMISSION_DIR / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(CODE_DIR))
import torch, torchvision, timm, platform
assert torch.cuda.is_available(), "Bật GPU trong Runtime → Change runtime type, rồi chạy lại."
print("Python:", platform.python_version(), "torch:", torch.__version__, "torchvision:", torchvision.__version__, "timm:", timm.__version__)
print("GPU:", torch.cuda.get_device_name())
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


## Lưu kết quả trên Google Drive

`MODE="full"`: 12 epoch, batch 32 dùng chung cho mọi backbone (giảm từ 64 của công thức gốc để phù hợp VRAM Colab; thay đổi này được ghi trong config). `MODE="smoke"`: 2 epoch để kiểm tra luồng, **không chạy test và không dùng để nộp kết quả**.

Giữ nguyên `SESSION_NAME` để khôi phục phiên chạy. Khi đổi cấu hình đã chạy, đặt tên phiên mới. Dữ liệu được giải nén vào `/content` để tránh đọc hàng nghìn ảnh trực tiếp từ Drive.


In [ ]:
drive.mount("/content/drive")
MODE = "full"  # full | smoke
BATCH_SIZE = 32  # Nếu thiếu VRAM: dùng 16 và SESSION_NAME mới, áp dụng cùng batch cho tất cả backbone.
SESSION_NAME = f"day2_{MODE}_batch{BATCH_SIZE}"
OUTPUT_DIR = Path("/content/drive/MyDrive/DeepWeedsLab") / SESSION_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Giữ mã thực sự dùng cùng kết quả để tái lập.
shutil.copytree(CODE_DIR, OUTPUT_DIR / "code", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
shutil.copy2(SUBMISSION_DIR / "requirements-colab.txt", OUTPUT_DIR)
subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=(OUTPUT_DIR / "environment-pip.txt").open("w"), check=True)
from colab_setup import prepare_data
IMAGES_DIR, LABELS_DIR = prepare_data(cache_dir="/content/drive/MyDrive/DeepWeedsLab/data-cache")
from train import Config, run
base = Config(images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR),
              out_dir=str(OUTPUT_DIR / "runs"), pred_dir=str(OUTPUT_DIR / "predictions"),
              curves_dir=str(OUTPUT_DIR / "curves"), batch_size=BATCH_SIZE,
              epochs=12 if MODE == "full" else 2)
print("Dữ liệu:", IMAGES_DIR, "Nhãn:", LABELS_DIR, "Kết quả:", OUTPUT_DIR)


## Kiểm tra cài đặt và các kỹ thuật dễ sai

Các kiểm tra focal γ=0, CutMix theo diện tích thực, gộp BN (cả nhánh residual), EMA, calibration và scheduler được chạy trên Colab trước khi huấn luyện.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", str(SUBMISSION_DIR / "tests"), "-o", "testpaths="], cwd=REPO_DIR, check=True)
# Bộ test gốc kiểm tra starter là khung bài tập; chạy ở process riêng để không lẫn module train.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], cwd=REPO_DIR, check=True)


## Bước 0 — EDA, kiểm tra split và overfit một batch

In [ ]:
from experiments import eda, pipeline_checks, compare_backbones, training_sweep, inference_sweep, final_runs, export_results
from IPython.display import display
split_stats = eda(base)
display(split_stats)
pipeline_checks(base)


## Bước 1 — So sánh 5 backbone

Cùng split, seed, batch, augmentation, loss, epoch và optimizer. Bảng có trọng số, tham số, GMAC, macro-F1/top-1 val, thời gian và độ trễ batch 1. GMAC dùng fvcore; các operation chưa được hỗ trợ được báo trong output.


In [ ]:
import pandas as pd
backbone_results = compare_backbones(base)
backbone_table = pd.DataFrame([{k:v for k,v in r.items() if k not in ("config", "latency")} | r["latency"] for r in backbone_results])
display(backbone_table[["exp_id", "backbone", "weight_tag", "params_M", "GMAC", "macro_f1_val", "top1_val", "train_seconds_per_epoch", "p95"]].sort_values("macro_f1_val", ascending=False))
# Ưu tiên macro-F1 val; dùng p95 làm tiêu chí phụ nếu hòa. Không đọc test.
backbone_winner = sorted(backbone_results, key=lambda r: (-r["macro_f1_val"], r["latency"]["p95"]))[0]
print("Backbone đi tiếp:", backbone_winner["backbone"], "theo macro-F1 val; p95:", backbone_winner["latency"]["p95"], "ms")


## Bước 2 — Ba trục huấn luyện và kết hợp

Trục A: finetune/frozen; B: basic/color; C: CE/label smoothing/focal. Mỗi thử nghiệm đơn chỉ đổi một yếu tố so với T00. T05 kết hợp các yếu tố cải thiện validation. Đây là sàng lọc một seed; kết luận về độ ổn định dựa trên vòng chung kết.


In [ ]:
from dataclasses import replace
training_base = replace(base, backbone=backbone_winner["backbone"])
training_results = training_sweep(training_base)
display(pd.DataFrame(training_results)[["exp_id", "axis", "change", "macro_f1_val", "delta"]])
training_winner = sorted(training_results, key=lambda r: (-r["macro_f1_val"], r["latency"]["p95"]))[0]
print("Công thức đi tiếp:", training_winner["exp_id"])


## Bước 3 — Suy luận trên validation và đo độ trễ

So với I00: TTA flip trung bình probability, TTA flip trung bình logit, temperature scaling, gộp BN, AMP. Đo thật: ≥10 warmup, 100 lần, synchronize trước/sau, batch 1, p50/p95/p99. Chọn trong các phương pháp có pipeline chung kết tích hợp; calibration khớp lại **trên val của từng seed**, trước khi đọc test.


In [ ]:
inference_results = inference_sweep(training_winner)
display(pd.DataFrame(inference_results)[["exp_id", "method", "K", "macro_f1_val", "ece_val", "p50", "p95", "p99", "dtype"]])
eligible = [r for r in inference_results if r["method"] in ("identity", "hflip_prob", "hflip_logit", "temperature")]
inference_winner = sorted(eligible, key=lambda r: (-r["macro_f1_val"], r["ece_val"], r["p95"]))[0]
print("Suy luận chung kết:", inference_winner["method"], "— chọn theo F1 val, sau đó ECE và p95")
# Đóng băng quyết định trước test.
selection = {"training": training_winner["config"], "inference": inference_winner, "backbone_selection": backbone_winner["exp_id"]}
selection_path = OUTPUT_DIR / "selection.json"
if selection_path.exists():
    assert json.loads(selection_path.read_text()) == selection, "Đã có quyết định chung kết khác; dùng SESSION_NAME mới."
else:
    selection_path.write_text(json.dumps(selection, indent=2))


## Bước 4 — Chung kết 3 seed, test một lần mỗi seed

T00 giữ công thức nền trên backbone đã chọn; F01 dùng công thức và suy luận đã chốt. Seed 0 của T00 tái sử dụng checkpoint từ vòng sàng lọc. File kết quả đã hoàn tất được đọc lại, không chạy lại test. Nếu phiên ngắt sau khi CSV test đã ghi nhưng summary chưa ghi, khôi phục từ CSV/logit đã lưu.


In [ ]:
final_results = []
if MODE == "full":
    final_results = final_runs(training_base, training_winner, inference_winner["method"])
    display(pd.DataFrame(final_results)[["exp_id", "seed", "macro_f1_val", "macro_f1_test", "top1_test", "ece_test"]])
else:
    print("Smoke mode: bỏ qua test và chung kết; đổi MODE=full khi chạy bài thực nghiệm.")


## Tính điểm bằng eval.py nguyên bản

In [ ]:
if final_results:
    common = ["--test-csv", str(LABELS_DIR / "test_subset0.csv"), "--labels", str(LABELS_DIR / "labels.csv"), "--out", str(OUTPUT_DIR / "eval_out")]
    for tag in ("F01", "T00"):
        subprocess.run([sys.executable, str(REPO_DIR / "eval.py"), "score", "--pred", str(OUTPUT_DIR / f"predictions/{tag}_seed*_test.csv"), "--tag", tag, *common], check=True)
    grade = [sys.executable, str(REPO_DIR / "eval.py"), "grade",
             "--final", str(OUTPUT_DIR / "predictions/F01_seed*_test.csv"),
             "--baseline", str(OUTPUT_DIR / "predictions/T00_seed*_test.csv"),
             "--final-val", str(OUTPUT_DIR / "predictions/F01_seed*_val.csv"),
             "--val-csv", str(LABELS_DIR / "val_subset0.csv"),
             "--latency-p95-ms", str(max(r["latency"]["p95"] for r in final_results if r["exp_id"] == "F01")), *common]
    if inference_winner["method"] == "temperature":
        grade += ["--uncal", str(OUTPUT_DIR / "predictions/F01uncal_seed*_test.csv")]
    subprocess.run(grade, check=True)


## Bước 5 — Excel, biểu đồ, báo cáo và tải sản phẩm

Xuất đủ 7 sheet; tính mean/sample std từ kết quả thật; tạo confusion matrix, ảnh lỗi và biểu đồ đánh đổi độ trễ. `report.md` là báo cáo tự động có số liệu và hạn chế; đọc ảnh lỗi để bổ sung phân tích của bạn. ZIP bài nộp loại checkpoint và dataset, chúng vẫn được giữ trên Drive để resume.


In [ ]:
xlsx = export_results(training_base, backbone_results, training_results, inference_results, final_results)
print("Đã lưu:", xlsx, "và", OUTPUT_DIR / "report.md")
archive_path = Path("/content/day2_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in ("code", "curves", "predictions", "eval_out"):
        for path in (OUTPUT_DIR / folder).rglob("*"):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path, path.relative_to(OUTPUT_DIR))
    for name in ("results.xlsx", "report.md", "all_results.json", "selection.json", "requirements-colab.txt", "environment-pip.txt"):
        path = OUTPUT_DIR / name
        if path.exists():
            archive.write(path, name)
    for path in (OUTPUT_DIR / "runs").rglob("*"):
        if path.is_file() and path.suffix in (".json", ".csv"):
            archive.write(path, path.relative_to(OUTPUT_DIR))
    archive.write(SUBMISSION_DIR / "README.md", "README.md")
files.download(str(archive_path))
